# Excel explorer

Select a source Excel file from `data/` to inspect its sheets and preview the first sheet.

In [4]:
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import display


# Local configuration: update these two paths only if their locations change.
DATA_DIR = Path(r"G:\coding_projects\rekenkamer\data")
PIPELINE_EXECUTABLE = Path(
    r"G:\coding_projects\workspaces\issues\3-move-excel-validation-from-the-analysis-workspace-into-the-implementatio\implementation\.venv\Scripts\rekenkamer-pipeline.exe"
)

data_dir = DATA_DIR

if not data_dir.is_dir():
    raise FileNotFoundError(
        f"Data directory does not exist: {data_dir}. "
        "Update DATA_DIR at the top of this notebook."
    )

excel_files = sorted(data_dir.glob('*.xlsx'))

if not excel_files:
    raise FileNotFoundError(f"No .xlsx files found in {data_dir}")

file_picker = widgets.Dropdown(
    options=[(file.name, file) for file in excel_files],
    description='Excel file:',
    layout=widgets.Layout(width='700px'),
)
sheet_picker = widgets.Dropdown(description='Sheet:', layout=widgets.Layout(width='400px'))
output = widgets.Output()


In [5]:
def show_preview(*_):
    """Load the selected sheet and display a small, safe preview."""
    selected_file = file_picker.value
    with output:
        output.clear_output(wait=True)
        try:
            frame = pd.read_excel(selected_file, sheet_name=sheet_picker.value, nrows=10)
            print(f"File: {selected_file.name}")
            print(f"Sheet: {sheet_picker.value} | Showing first {len(frame)} rows")
            display(frame)
        except Exception as error:
            print(f"Could not read the selected sheet: {error}")


def update_sheets(*_):
    """Update the sheet dropdown after choosing another workbook."""
    try:
        workbook = pd.ExcelFile(file_picker.value)
        sheet_picker.options = workbook.sheet_names
        sheet_picker.value = workbook.sheet_names[0]
    except Exception as error:
        with output:
            output.clear_output(wait=True)
            print(f"Could not open the selected workbook: {error}")


file_picker.observe(update_sheets, names='value')
sheet_picker.observe(show_preview, names='value')
update_sheets()
display(widgets.VBox([file_picker, sheet_picker, output]))


In [6]:
# Export the selected workbook's schema and validate it with the implementation CLI.
import json
import subprocess


def logical_type(series: pd.Series) -> str:
    """Map pandas dtypes to the validation contract's stable logical types."""
    if pd.api.types.is_bool_dtype(series):
        return "boolean"
    if pd.api.types.is_integer_dtype(series):
        return "integer"
    if pd.api.types.is_numeric_dtype(series):
        return "number"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "date"
    return "string"


workbook_path = Path(file_picker.value)
workbook = pd.read_excel(workbook_path, sheet_name=None)
schema = {
    "version": "1",
    "sheets": {
        sheet_name: {
            "columns": {str(column): logical_type(frame[column]) for column in frame.columns}
        }
        for sheet_name, frame in workbook.items()
    },
}

generated_dir = data_dir / "generated"
generated_dir.mkdir(parents=True, exist_ok=True)
schema_file = generated_dir / "original_schema.json"
schema_file.write_text(json.dumps(schema, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"Schema saved to: {schema_file}")

pipeline = PIPELINE_EXECUTABLE
if not pipeline.is_file():
    raise RuntimeError(
        f"Implementation CLI was not found: {pipeline}. "
        "Update PIPELINE_EXECUTABLE at the top of this notebook."
    )

report_file = generated_dir / "reports" / f"{workbook_path.stem}.validation.json"
command = [str(pipeline), "validate", str(workbook_path), "--schema", str(schema_file), "--output", str(report_file)]
completed = subprocess.run(command, text=True, capture_output=True, check=False)
print(completed.stdout, end="")
if completed.stderr:
    print(completed.stderr, end="")
if completed.returncode == 2:
    raise RuntimeError("The validator could not run; see the output above.")

report = json.loads(report_file.read_text(encoding="utf-8"))
display(pd.DataFrame([{
    "file": report["source_filename"],
    "status": report["status"],
    "issues": " | ".join(issue["message"] for issue in report["issues"]),
}]))


Schema saved to: G:\coding_projects\rekenkamer\data\generated\original_schema.json
passed: Algemene Rekenkamer - Casus Data Engineer v1.xlsx -> G:\coding_projects\rekenkamer\data\generated\reports\Algemene Rekenkamer - Casus Data Engineer v1.validation.json


,file,status,issues
0,Algemene Rekenkamer - Casus Data Engineer v1.xlsx,passed,
